**Week 10 — Dummy variable, Interaction, Polynomial, Diagnostics (Studentized residual, Cook's D) และ VIF**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:**
  - ข้อมูลลูกค้า: `cust = pd.read_csv('biz_customers.csv')` (1203 แถว)
  - ข้อมูลแคมเปญ: `camp = pd.read_csv('biz_campaigns.csv')` (240 แคมเปญ)
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์
- **แบบจำลอง OLS:** ทุกโมเดลมี intercept และประมาณด้วย `statsmodels.formula.api.smf.ols`
- **ตัวแปรกลุ่มและกลุ่มอ้างอิง:** ใช้ `C(..., Treatment(reference=...))` เพื่อกำหนดกลุ่มอ้างอิงอย่างชัดเจน
- **การตรวจวินิจฉัย (Diagnostics):**
  - Externally Studentized Residual: `model.get_influence().resid_studentized_external`
  - Cook's Distance: `model.get_influence().cooks_distance[0]`
  - VIF: `variance_inflation_factor` จาก `statsmodels.stats.outliers_influence`

### รายละเอียดไฟล์ข้อมูล

**1. `biz_customers.csv`** (1203 แถว แถวละ 1 คน)
- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `channel`: ช่องทางที่ซื้อบ่อย (`online` = ออนไลน์, `store` = หน้าร้าน)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)

**2. `biz_campaigns.csv`** (240 แถว แถวละ 1 แคมเปญ)
- `tv`: งบโฆษณาทางโทรทัศน์ (พันบาท)
- `radio`: งบโฆษณาทางวิทยุ (พันบาท)
- `social`: งบโฆษณาทางโซเชียลมีเดีย (พันบาท)
- `email`: งบโฆษณาทางอีเมล (พันบาท)
- `season`: ไตรมาสของแคมเปญ (`Q1`–`Q4`)
- `sales`: ยอดขายของแคมเปญ (พันชิ้น)
- `total_budget`: งบโฆษณารวมทุกสื่อ (= tv + radio + social + email)

**โจทย์ข้อ 10.1:** ฝึกทำนาย `spend_k` จาก `tenure_months` และ `channel` (ใช้ dummy variable โดยให้ `online` เป็นกลุ่มอ้างอิง; `spend_k ~ tenure_months + C(channel)`; มี intercept) ใช้ลูกค้าทั้ง 1203 คนในไฟล์ `biz_customers.csv` ค่าสัมประสิทธิ์ของกลุ่ม `store` = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลลูกค้า
cust = pd.read_csv('biz_customers.csv')

# 2. Fit โมเดลโดยให้ online เป็นกลุ่มอ้างอิง
formula = 'spend_k ~ tenure_months + C(channel, Treatment(reference="online"))'
model = smf.ols(formula, data=cust).fit()
print(model.params)

# 3. ค่าสัมประสิทธิ์ของกลุ่ม store
coef_store = model.params['C(channel, Treatment(reference="online"))[T.store]']
print(f"สัมประสิทธิ์กลุ่ม store: {coef_store:.4f}")
print(f"คำตอบข้อ 10.1: {coef_store:.4f}")


Intercept                                                3.420878
C(channel, Treatment(reference="online"))[T.store]      -0.487352
tenure_months                                            0.007727
dtype: float64
สัมประสิทธิ์กลุ่ม store: -0.4874
คำตอบข้อ 10.1: -0.4874


**โจทย์ข้อ 10.2:** กรณีศึกษาธุรกิจสมัครสมาชิกต้องการดูว่าอายุความเป็นลูกค้ามีผลต่อยอดซื้อต่อเดือน (`spend_k`) หรือไม่ หลังควบคุมรายได้ (`income_k`) จึงแบ่ง `tenure_months` (ระยะเวลาที่เป็นลูกค้า) เป็น 3 กลุ่มด้วย `pd.cut(tenure_months, bins=[0, 18, 48, 200], labels=['New', 'Mid', 'Long'])` (ขอบขวาปิด: $\le 18$ = New, $18 < \dots \le 48$ = Mid, $> 48$ = Long) แล้ว fit `spend_k ~ income_k + C(tenure_group)` ด้วยลูกค้าทั้ง 1203 คนใน `biz_customers.csv` โดยให้ `New` เป็นกลุ่มอ้างอิง ค่าสัมประสิทธิ์ของกลุ่ม `Long` = ? (ทศนิยม 4 ตำแหน่ง ในหน่วยพันบาทต่อเดือน)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. แบ่งกลุ่ม tenure_months ออกเป็น 3 ช่วง (ขอบขวาปิด)
cust = pd.read_csv('biz_customers.csv')
cust['tenure_group'] = pd.cut(cust['tenure_months'], bins=[0, 18, 48, 200], labels=['New', 'Mid', 'Long'])
print("จำนวนลูกค้าในแต่ละกลุ่ม tenure:")
print(cust['tenure_group'].value_counts())

# 2. Fit โมเดลโดยให้ New เป็นกลุ่มอ้างอิง
formula = 'spend_k ~ income_k + C(tenure_group, Treatment(reference="New"))'
model = smf.ols(formula, data=cust).fit()
print(model.params)

# 3. ค่าสัมประสิทธิ์ของกลุ่ม Long
coef_long = model.params['C(tenure_group, Treatment(reference="New"))[T.Long]']
print(f"สัมประสิทธิ์กลุ่ม Long: {coef_long:.4f}")
print(f"คำตอบข้อ 10.2: {coef_long:.4f}")


จำนวนลูกค้าในแต่ละกลุ่ม tenure:
tenure_group
Mid     608
New     398
Long    197
Name: count, dtype: int64
Intercept                                              1.895550
C(tenure_group, Treatment(reference="New"))[T.Mid]     0.102113
C(tenure_group, Treatment(reference="New"))[T.Long]    0.399793
income_k                                               0.048410
dtype: float64
สัมประสิทธิ์กลุ่ม Long: 0.3998
คำตอบข้อ 10.2: 0.3998


**โจทย์ข้อ 10.3:** ฝึก fit โมเดลที่มี interaction ระหว่างช่องทางกับ `income_k` : `spend_k ~ income_k + C(channel) + income_k:C(channel)` (กลุ่มอ้างอิงคือ `online`) ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` **ความชันของ `income_k` ในกลุ่ม store** (การเปลี่ยนแปลงของ `spend_k` เมื่อ `income_k` เพิ่ม 1 หน่วย เฉพาะลูกค้ากลุ่ม store) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล Interaction
cust = pd.read_csv('biz_customers.csv')
formula = 'spend_k ~ income_k + C(channel, Treatment(reference="online")) + income_k:C(channel, Treatment(reference="online"))'
model = smf.ols(formula, data=cust).fit()
print(model.params)

# 2. คำนวณความชันของ income_k ในกลุ่ม store
# ความชันกลุ่ม store = beta_income_k + beta_interaction
beta_income = model.params['income_k']
beta_interaction = model.params['income_k:C(channel, Treatment(reference="online"))[T.store]']
slope_store = beta_income + beta_interaction
print(f"ความชันกลุ่ม online: {beta_income:.4f}")
print(f"พจน์ interaction: {beta_interaction:.4f}")
print(f"ความชันกลุ่ม store: {slope_store:.4f}")
print(f"คำตอบข้อ 10.3: {slope_store:.4f}")


Intercept                                                                     2.109703
C(channel, Treatment(reference="online"))[T.store]                           -0.259468
income_k                                                                      0.051448
income_k:C(channel, Treatment(reference="online"))[T.store]                  -0.008240
dtype: float64
ความชันกลุ่ม online: 0.0514
พจน์ interaction: -0.0082
ความชันกลุ่ม store: 0.0432
คำตอบข้อ 10.3: 0.0432


**โจทย์ข้อ 10.4:** ฝึก fit โมเดลพหุนามของ `age` ดีกรี 2 เพื่อทำนาย `spend_k` : `spend_k ~ income_k + age + I(age**2)` (มี intercept) ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ค่า $R^2$ ของโมเดล = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit Polynomial Regression ดีกรี 2 สำหรับ age
cust = pd.read_csv('biz_customers.csv')
formula = 'spend_k ~ income_k + age + I(age**2)'
model = smf.ols(formula, data=cust).fit()

# 2. ดึงค่า R-squared
r2 = model.rsquared
print(f"R-squared: {r2:.4f}")
print(f"คำตอบข้อ 10.4: {r2:.4f}")


R-squared: 0.2263
คำตอบข้อ 10.4: 0.2263


**โจทย์ข้อ 10.5:** ฝึกในโมเดลที่ใช้ `tv`, `radio`, `email`, `total_budget` เป็นตัวทำนาย (มี intercept) ใช้แคมเปญทั้ง 240 แคมเปญในไฟล์ `biz_campaigns.csv` (`total_budget` คืองบโฆษณารวมทุกสื่อ = tv + radio + social + email ; โมเดลนี้ไม่ใส่ social) ค่า **VIF ของ `total_budget`** = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. เตรียมเมทริกซ์ตัวแปรทำนาย (พร้อม intercept)
camp = pd.read_csv('biz_campaigns.csv')
X = camp[['tv', 'radio', 'email', 'total_budget']]
X_with_const = sm.add_constant(X)

# 2. คำนวณ VIF ของ total_budget (คอลัมน์ลำดับที่ 4 ใน X_with_const)
idx_total = list(X_with_const.columns).index('total_budget')
vif_tb = variance_inflation_factor(X_with_const.values, idx_total)
print(f"VIF ของ total_budget: {vif_tb:.2f}")
print(f"คำตอบข้อ 10.5: {vif_tb:.2f}")


VIF ของ total_budget: 24.38
คำตอบข้อ 10.5: 24.38


**โจทย์ข้อ 10.6:** กรณีศึกษาทีมวิเคราะห์ของเอเจนซีโฆษณาทำนายยอดขายของแคมเปญ (`sales`) จากงบโฆษณาทางโทรทัศน์ (`tv`), ทางวิทยุ (`radio`), งบโฆษณาทางโซเชียลมีเดีย (`social`) และงบโฆษณารวมทุกสื่อ (`total_budget` = tv + radio + social + email ; โมเดลนี้ไม่ใส่ email) (มี intercept, ใช้แคมเปญทั้ง 240 แคมเปญใน `biz_campaigns.csv`) แล้วพบว่า `tv` มี standard error ใหญ่ผิดปกติ จึงสงสัยว่าเกิด multicollinearity และตัด `total_budget` ออกจากโมเดล (ที่เหลือ tv, radio, social) อัตราส่วน **(standard error ของสัมประสิทธิ์ tv ในโมเดลที่มี total_budget) ÷ (standard error ของสัมประสิทธิ์ tv ในโมเดลหลังตัด total_budget)** = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. โมเดลที่มี total_budget
camp = pd.read_csv('biz_campaigns.csv')
m1 = smf.ols('sales ~ tv + radio + social + total_budget', data=camp).fit()
se_tv_m1 = m1.bse['tv']

# 2. โมเดลหลังตัด total_budget ออก
m2 = smf.ols('sales ~ tv + radio + social', data=camp).fit()
se_tv_m2 = m2.bse['tv']

# 3. คำนวณอัตราส่วน Standard Error
ratio = se_tv_m1 / se_tv_m2
print(f"SE(tv) โมเดลมี total_budget: {se_tv_m1:.6f}")
print(f"SE(tv) โมเดลตัด total_budget: {se_tv_m2:.6f}")
print(f"อัตราส่วน SE: {ratio:.3f}")
print(f"คำตอบข้อ 10.6: {ratio:.3f}")


SE(tv) โมเดลมี total_budget: 0.016271
SE(tv) โมเดลตัด total_budget: 0.001564
อัตราส่วน SE: 10.401
คำตอบข้อ 10.6: 10.401


**โจทย์ข้อ 10.7:** ฝึกทำนาย `sales` จาก `tv`, `radio`, `social` (มี intercept) ใช้แคมเปญทั้ง 240 แคมเปญใน `biz_campaigns.csv` ค่า **Cook's distance ที่มากที่สุด**ในบรรดาแคมเปญทั้งหมด = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล OLS
camp = pd.read_csv('biz_campaigns.csv')
model = smf.ols('sales ~ tv + radio + social', data=camp).fit()

# 2. คำนวณ Cook's distance
influence = model.get_influence()
cooks_d = influence.cooks_distance[0]
max_cooks = np.max(cooks_d)
print(f"Cook's distance สูงสุด: {max_cooks:.4f}")
print(f"คำตอบข้อ 10.7: {max_cooks:.4f}")


Cook's distance สูงสุด: 0.0583
คำตอบข้อ 10.7: 0.0583


**โจทย์ข้อ 10.8:** กรณีศึกษาฝ่ายควบคุมคุณภาพข้อมูลของศูนย์กระจายสินค้าสร้างโมเดลทำนาย `spend_k` (ยอดซื้อต่อเดือน) จาก รายได้ (`income_k`), จำนวนครั้งที่เข้าเว็บ/แอป (`web_visits`), การเป็นสมาชิกสะสมแต้ม (`loyalty_member`) (มี intercept, ใช้บัญชีลูกค้าทั้ง 1203 คนใน `biz_customers.csv`) แล้วต้องการตรวจทานบันทึกของลูกค้าที่เป็น outlier มากที่สุด โดยดูจาก **externally studentized residual ที่มีค่าสัมบูรณ์สูงสุด** ลูกค้ารายนั้นอยู่ที่**แถวลำดับใด** (ดัชนีแถวนับจาก 0 ตามลำดับในไฟล์) = ? (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล OLS
cust = pd.read_csv('biz_customers.csv')
model = smf.ols('spend_k ~ income_k + web_visits + loyalty_member', data=cust).fit()

# 2. คำนวณ externally studentized residual
influence = model.get_influence()
stud_resid = influence.resid_studentized_external

# 3. หาตำแหน่งแถวที่มีค่าสัมบูรณ์สูงสุด
worst_row = int(np.argmax(np.abs(stud_resid)))
print(f"ค่าสัมบูรณ์สูงสุด: {np.abs(stud_resid[worst_row]):.4f}")
print(f"หมายเลขแถวที่เป็น outlier สูงสุด: {worst_row}")
print(f"คำตอบข้อ 10.8: {worst_row}")


ค่าสัมบูรณ์สูงสุด: 5.1113
หมายเลขแถวที่เป็น outlier สูงสุด: 1087
คำตอบข้อ 10.8: 1087


**โจทย์ข้อ 10.9:** กรณีศึกษาทีมวิเคราะห์ของบริษัทที่ปรึกษาด้านการตลาดสงสัยว่าความสัมพันธ์ระหว่าง `tv` (งบโฆษณาทางโทรทัศน์) กับ `sales` (ยอดขายของแคมเปญ) อาจขึ้นกับระดับของ `radio` (งบโฆษณาทางวิทยุ) จึงเทียบโมเดล additive `sales ~ tv + radio` กับโมเดลที่เพิ่ม interaction `sales ~ tv + radio + tv:radio` (มี intercept ทั้งคู่ ใช้แคมเปญทั้ง 240 แคมเปญใน `biz_campaigns.csv`) ค่า **AIC ของโมเดล additive ลบด้วย AIC ของโมเดลที่มี interaction** (ค่า `aic` ที่ statsmodels รายงาน; ค่าบวกหมายถึงโมเดลที่มี interaction ดีกว่า) = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล additive และโมเดล interaction
camp = pd.read_csv('biz_campaigns.csv')
m_add = smf.ols('sales ~ tv + radio', data=camp).fit()
m_int = smf.ols('sales ~ tv + radio + tv:radio', data=camp).fit()

# 2. คำนวณผลต่าง AIC: AIC(additive) - AIC(interaction)
aic_diff = m_add.aic - m_int.aic
print(f"AIC โมเดล Additive: {m_add.aic:.2f}")
print(f"AIC โมเดล Interaction: {m_int.aic:.2f}")
print(f"ผลต่าง AIC: {aic_diff:.2f}")
print(f"คำตอบข้อ 10.9: {aic_diff:.2f}")


AIC โมเดล Additive: 1037.41
AIC โมเดล Interaction: 929.65
ผลต่าง AIC: 107.76
คำตอบข้อ 10.9: 107.76


**โจทย์ข้อ 10.10:** กรณีศึกษาผู้จัดการฝ่ายการตลาดของแบรนด์เครื่องใช้ไฟฟ้าสงสัยว่าโทรทัศน์กับวิทยุเสริมกันในการขาย และยอดขายต่างกันไปตามไตรมาส จึง fit โมเดล `sales ~ tv + radio + tv:radio + C(season)` (กลุ่มอ้างอิงคือ `Q1`, มี intercept, ใช้แคมเปญทั้ง 240 แคมเปญใน `biz_campaigns.csv`) แล้วใช้ทำนายแคมเปญที่วางแผนไว้ใน**ไตรมาส Q4 ที่มีงบโฆษณาทางโทรทัศน์ (tv) = 233 และทางวิทยุ (radio) = 33** (พันบาท) ค่า `sales` ที่โมเดลทำนาย = ? พันชิ้น (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดลที่มี interaction และฤดูกาล
camp = pd.read_csv('biz_campaigns.csv')
formula = 'sales ~ tv + radio + tv:radio + C(season, Treatment(reference="Q1"))'
model = smf.ols(formula, data=camp).fit()
print(model.params)

# 2. ทำนายแคมเปญใหม่: ไตรมาส Q4, tv = 233, radio = 33
target_campaign = pd.DataFrame([{'tv': 233.0, 'radio': 33.0, 'season': 'Q4'}])
pred_sales = model.predict(target_campaign)[0]
print(f"ยอดขายที่ทำนายได้: {pred_sales:.2f}")
print(f"คำตอบข้อ 10.10: {pred_sales:.2f}")


Intercept                                     2.621720
C(season, Treatment(reference="Q1"))[T.Q2]    0.679440
C(season, Treatment(reference="Q1"))[T.Q3]   -0.470831
C(season, Treatment(reference="Q1"))[T.Q4]    1.116442
tv                                            0.047095
radio                                         0.187513
tv:radio                                      0.001065
dtype: float64
ยอดขายที่ทำนายได้: 29.09
คำตอบข้อ 10.10: 29.09


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 10

---

### 1. การแปลงตัวแปรกลุ่มเป็นตัวแปรดัมมี่ (Dummy Variables)

เมื่อตัวแปรเชิงคุณภาพมี $K$ ระดับ จะสร้างตัวแปรดัมมี่ได้ $K - 1$ ตัว โดยระดับที่เหลือจะถูกกำหนดเป็น **ระดับอ้างอิง (Baseline / Reference Group)** เพื่อหลีกเลี่ยง Dummy Variable Trap (Multicollinearity สมบูรณ์):
$$Y = \beta_0 + \beta_1 D_{\text{store}} + \dots$$
- $\beta_0$: คือค่าเฉลี่ยของกลุ่มอ้างอิง (`online`)
- $\beta_1$: คือผลต่างระหว่างค่าเฉลี่ยของกลุ่ม `store` เทียบกับกลุ่มอ้างอิง

---

### 2. พจน์ปฏิสัมพันธ์ (Interaction Terms)

พจน์ปฏิสัมพันธ์ช่วยให้โมเดลสามารถจับความสัมพันธ์แบบ Synergy หรือผลกระทบของตัวแปรหนึ่งที่ขึ้นอยู่กับอีกตัวแปรหนึ่งได้:
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 (X_1 \times X_2) + \epsilon$$
- **ความชันของ $X_1$:** $\frac{\partial Y}{\partial X_1} = \beta_1 + \beta_3 X_2$ (ขึ้นกับระดับของ $X_2$)
- **ระหว่างตัวแปรต่อเนื่องกับตัวแปรกลุ่ม:** ช่วยให้ความชันของตัวแปรต่อเนื่องแยกตามกลุ่มได้แตกต่างกัน (Differing Slopes)

---

### 3. ปัญหาภาวะร่วมเส้นตรงพหุคูณ (Multicollinearity) และ VIF

เมื่อตัวแปรทำนายในโมเดลมีความสัมพันธ์เชิงเส้นระหว่างกันสูง ค่า Standard Error ของสัมประสิทธิ์จะขยายใหญ่ขึ้นอย่างมาก ทำให้ค่า $t$-statistic ต่ำและแปลความหมายได้ยาก:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
โดย $R_j^2$ คือ $R^2$ จากการถดถอยตัวแปร $X_j$ ด้วยตัวแปรทำนายตัวอื่นๆ ทั้งหมดในโมเดล:
- $\text{VIF} = 1$: ไม่มีความสัมพันธ์เชิงเส้นเลย
- $\text{VIF} > 5$ หรือ $10$: บ่งชี้ว่าเกิดปัญหาร่วมเส้นตรงพหุคูณอย่างรุนแรง (เช่น `total_budget` ที่มี VIF สูงถึง 24.38)
- การตัดตัวแปรที่ซ้ำซ้อนออกจะช่วยลด SE ของสัมประสิทธิ์ตัวอื่นลงได้อย่างมีนัยสำคัญ (ในข้อ 10.6 ลดลงถึง 10.4 เท่า)

---

### 4. การตรวจวินิจฉัยโมเดล (Regression Diagnostics)

1. **Externally Studentized Residuals ($t_i$):**
   $$t_i = \frac{e_i}{\text{SE}_{(i)} \sqrt{1 - h_{ii}}}$$
   วัดว่าจุดข้อมูล $y_i$ เป็น Outlier ผิดปกติหรือไม่ โดยคำนวณส่วนเบี่ยงเบนมาตรฐานหลังจากตัดจุดที่ $i$ ออก จุดที่มี $|t_i| > 3$ จัดเป็น Outlier ที่มีนัยสำคัญ

2. **Cook's Distance ($D_i$):**
   $$D_i = \frac{\sum_{j=1}^n (\hat{y}_j - \hat{y}_{j(i)})^2}{p \cdot \text{MSE}}$$
   วัดอิทธิพล (Influence) ของจุดข้อมูลที่ $i$ ต่อค่าพยากรณ์ทั้งหมดของโมเดล จุดที่มี $D_i$ สูงมักเป็นจุดที่มีทั้ง Residual สูงและ Leverage ($h_{ii}$) สูง